## How does a focus on professionalism affect engagement?

### 1. Dumbbell plot

The plot in this part is a dumbbell plot comparing metrics for sponsored vs nonsponsored videos within sponsored channels. The first step is to gather data from different files in order to make the plot, and the code used to do this is found in [./process/rq3_processing/rq3_build_data_dumbbell.py](./process/rq3_processing/rq3_build_data_dumbbell.py), with the main steps described below.

The aim here is to get information about every video within channels that are at least registered five times in SponsorBlock for having in-video sponsor segment. This will be used to create a plot comparing sponsored vs nonsponsored video metrics (like/dislike ratio, duration and number of views).

First, only channels with at least five videos in SponsorBlock are kept:
```python
    df_gt5 = df_sc[df_sc["video_ids"].apply(len) >= 5]
```

Then, to collect the metadata of the videos of interest, the metadata file is read in chunks due to its large size, and only a subset of the metrics are gathered for the plot:
```python
    meta_chunks = pd.read_json(
            METADATA_FILE,
            lines=True,
            chunksize=1_000_000,
            compression="gzip"
        )

    ...

    subset = chunk.loc[mask, ["categories",
                              "channel_id",
                              "display_id",
                              "dislike_count",
                              "duration",
                              "like_count",
                              "view_count"
                              ]
                        ]
```

Finally, the like/dislike ratio, defined as $\frac{\text{likes}}{\text{likes+dislikes}} \cdot 100$, is computed. In addition, the videos are specified as "sponsored" or not:

```python
    dislikes = df_step1["dislike_count"]
    likes = df_step1["like_count"]
        
    df_step1["like_dislike_ratio"] = (likes/(likes+dislikes))*100
        
    df_step1["sponsored"] = df_step1["display_id"].isin(sponsored_videos_ids)
```


Then, the aim is to get a json file with all the values used for plotting (the code to do that is located in [./analysis/rq3_analysis/rq3_dumbbell_data.py](./analysis/rq3_analysis/rq3_dumbbell_data.py)). 

Using aggregation, the median of the metrics are computed per channel in each category and then the median of these previously computed medians are calculated (the procedure is more detailed on the website):
```python
    per_channel = (
        df.groupby(["categories", "channel_id"])[metric_col]
        .median()
        .reset_index()
    )

    agg = (
        per_channel.groupby("categories")[metric_col]
        .median()
        .sort_values(ascending=False)
    )
```

Also, the sponsored and nonsponsored videos are separated,

```python
df_non = df[~df["sponsored"]].copy()  
df_spon = df[df["sponsored"]].copy() 
```
and are used to obtain `ratio_non`, `ratio_spon`, `duration_non`, `duration_spon`, `views_non` and `views_spon`. For instance, `duration_spon` corresponds to the computed duration medians in sponsored videos for each categories.

At the end, the values used for the plot are saved in the following dictionary:
```python
    second_plot_data = {
        "categories": cat_order,
        "metrics": {
            "ratio": {
                "non_sponsored": ratio_non,
                "sponsored": ratio_spon,
            },
            "duration": {
                "non_sponsored": duration_non,
                "sponsored": duration_spon,
            },
            "views": {
                "non_sponsored": views_non,
                "sponsored": views_spon,
            },
        },
    }
```
These saved values can be found in [./analysis/rq3_analysis/rq3_dumbbell_data.json](./analysis/rq3_analysis/rq3_dumbbell_data.json) and are directly used to produce the dumbell plot in [./analysis/rq3_analysis/rq3_dumbbell.py](./analysis/rq3_analysis/rq3_dumbbell.py).


### 2. Plot with delta_views and delta_subs, centered around first sponsorship appearance


This plot also requires gathering data from multiple files. In [./process/rq3_processing/rq3_build_data_delta_views_subs.py](./process/rq3_processing/rq3_build_data_delta_views_subs.py), the preprocessing steps are done, with the most important milestones described here.

The aim is to get time-series information about channels that are at least registered once in SponsorBlock for having in-video sponsor segment. This will be used to create a plot showing the changes happening to the channels stats (weekly change in views and subscribers) around the first-ever sponsor appearance.

First of all, as with the first plot, the metadata file used to get information about the videos is read in chunks, but this time, only `channel_id`, `display_id` and `upload_date` are kept for the videos of interest:
```python
    subset = chunk.loc[mask, ["channel_id", "display_id", "upload_date"]]
```

The upload date is converted to datetime type (for further computation):
```python
    df_meta_sponsor["upload_date"] = pd.to_datetime(df_meta_sponsor["upload_date"])
```

Then, the time-series data is gathered for the channels of interests and the same convertion to datetime type is applied to `datetime`, which corresponds to every week data is gathered for a channel in the time-series:
```python
    df_ts = pd.read_csv(
        TIMESERIES_FILE, 
        sep="\t", 
        compression="gzip",
        usecols=["channel", "datetime", "views", "delta_views", "subs", "delta_subs"]
        )

    df_ts["datetime"] = pd.to_datetime(df_ts["datetime"])
```

Finally, the cornerstone of this plot is getting the weeks before and after the first sponsor appearance. To do that, the first step is to find the earliest sponsored video within each channel in a new column `sponsor_start`. Then, this value is substracted from each `datetime` in the time-series. This difference is then converted to days and eventually the floor division with 7 is taken to compute the relative weeks from the first sponsor. For example, in that case week 0 is defined as the time window from 0 to 6 days after the first sponsor appearance (inclusive).
```python
    min_per_channel = (
        df_step2[["channel_id", "upload_date"]]
        .groupby("channel_id")["upload_date"]
        .min()                      
    )    

    df_step3["sponsor_start"] = df_step3["channel_id"].map(min_per_channel)

    df_step3["delta_weeks"] = (
    (df_step3["datetime"] - df_step3["sponsor_start"]).dt.days // 7
    )
```


The code for plotting is located in [./analysis/rq3_analysis/rq3_delta_views_subs.py](./analysis/rq3_analysis/rq3_delta_views_subs.py), and below is the presentation of the main steps.

Only the 26 weeks before and after the first sponsor are kept to cover a full year:
```python
    df_keep = df[df["delta_weeks"].between(-26, 26, inclusive="both")].copy()
```

Then, the logarithm of the metric (either `delta_views` or `delta_subs`) is taken. More specifically, numpy's `log1p` (computing log(1 + x)) is used, making sure any possible undefined logarithm computations are avoided. This step is followed by the aggregation across channels considering the relative weeks to the first sponsor (`delta_weeks`), where new columns are created to get the mean, standard deviation and count for every relative week.
```python
    log_col = f"log_{metric_col}"
    df_temp[log_col] = np.log1p(df_temp[metric_col].clip(lower=0))

    df_plot = (
        df_temp.groupby("delta_weeks", as_index=False)
        .agg(
            mean_log_val=(log_col, "mean"),
            std_log_val=(log_col, "std"),
            n=(log_col, "size")
        )
        .sort_values("delta_weeks")
    )
```

To quantify uncertainty around the calculated means, 95% confidence intervals are computed using the following formula: 
$$
CIs = \bar{x} \pm SE,
$$
where $\bar{x} = \frac{1}{n} \sum x_i$ is the sample mean and $SE = \frac{\text{standard deviation}}{\sqrt{\text{number of elements}}} = \frac{\sigma}{\sqrt n}$ is the standard error.
```python
    df_plot["se"] = df_plot["std_log_val"] / np.sqrt(df_plot["n"])
    df_plot["ci_lower"] = df_plot["mean_log_val"] - (1.96 * df_plot["se"])
    df_plot["ci_upper"] = df_plot["mean_log_val"] + (1.96 * df_plot["se"])
```

Finally, all the values used in the plot are saved in the following dictionary:
```python
    {
        "delta_weeks": df_plot["delta_weeks"].tolist(),
        "mean_log_val": df_plot["mean_log_val"].tolist(),
        "ci_lower": df_plot["ci_lower"].tolist(),
        "ci_upper": df_plot["ci_upper"].tolist(),
    }
```

and these are directly used for plotting within the same file.
